# 24 — Chat with Teacher Model

Interactive REPL to test the **trained 30B teacher** before running distillation (NB21).

Distillation in `21_distillation` uses the teacher to generate ~5K outputs that train the smaller 8B student.
If the teacher is broken, those outputs are useless and the student inherits the breakage.
This notebook lets you sanity-check the teacher first.

## What it does

- Loads the best available teacher: DPO (NB18) > iterative (NB20) > fine-tune (NB17) > warm-start (NB05)
- Shows which checkpoint was selected and its size
- Opens an interactive chat loop with auto-`aro check` validation on generated code
- Runs a fixed probe set so you can compare teachers across runs

## When to use

Run this **before NB21** to confirm the teacher actually produces sensible ARO.
If the teacher fails the probes here, fix the upstream training (NB17/NB18/NB20) before
spending an hour generating distillation data with a bad model.

**Input:**  best fused checkpoint from `models/dpo/`, `models/iterative/`, `models/finetune/`, or `data/adapters/warm_start/`
**Output:**  console interaction (no files) + `run/<date>/24_chat_teacher.png` probe summary


## Setup

In [ ]:
import sys, importlib, re, subprocess, tempfile, json
from pathlib import Path

_cfg_dir = Path('.').resolve()
if str(_cfg_dir) not in sys.path:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *

MAX_TOKENS  = 1024
TEMPERATURE = 0.3
TOP_P       = 0.9

print(f'Base model:      {MODEL_ID}')
print(f'Models dir:      {MODELS_DIR}')

## Select the best teacher checkpoint

Same priority as NB21 (`find_best_teacher`):
DPO > iterative (highest round) > fine-tune (highest round) > warm-start adapter > base.

In [ ]:
def find_best_teacher():
    """Mirrors the selector in NB12 cell distill_02."""
    import re as _re
    def _highest_fused(parent, label):
        if not parent.exists(): return None, None
        rounds = sorted(
            [d for d in parent.glob('round_*/fused') if (d / 'config.json').exists()],
            key=lambda p: int(_re.search(r'round_(\d+)', str(p)).group(1))
        )
        if rounds:
            best = rounds[-1]
            return str(best), label
        return None, None

    # 1. DPO (NB12)
    dpo = MODELS_DIR / 'dpo' / 'fused'
    if dpo.exists() and (dpo / 'config.json').exists():
        return str(dpo), 'dpo (NB12)'

    # 2. Iterative (NB12)
    p, lbl = _highest_fused(ITERATIVE_MODELS_DIR, 'iterative (NB12)')
    if p: return p, lbl

    # 3. Fine-tune (NB12)
    p, lbl = _highest_fused(FINETUNE_MODELS_DIR, 'finetune (NB12)')
    if p: return p, lbl

    # 4. Warm-start adapter (NB05) — fused on the fly via load_model
    warm = resolve_warm_adapter()
    if warm:
        return None, 'warm-start adapter (NB05)'

    return None, 'base model (no fine-tune)'


def _dir_size_gb(path):
    if not path or not Path(path).exists():
        return 0.0
    return sum(f.stat().st_size for f in Path(path).rglob('*') if f.is_file()) / 1e9


teacher_path, teacher_source = find_best_teacher()

print(f'Teacher source:  {teacher_source}')
if teacher_path:
    print(f'Teacher path:    {teacher_path}')
    print(f'Teacher size:    {_dir_size_gb(teacher_path):.2f} GB')
else:
    print(f'Teacher path:    (loading {MODEL_ID} with warm-start adapter)')

## Load teacher model

In [ ]:
load_fn, generate_fn, make_sampler_fn = ensure_mlx_lm()

if teacher_path:
    print(f'Loading teacher from {teacher_path}...')
    model, tokenizer = load_fn(teacher_path)
else:
    print(f'Loading {MODEL_ID} with warm-start adapter...')
    model, tokenizer, _, _, _ = load_model(with_adapter=True)

kb            = load_knowledge()
SYSTEM_PROMPT = build_system_prompt(kb)

print(f'\nTeacher loaded.')
print(f'System prompt: {len(SYSTEM_PROMPT)} chars')

## Helpers

In [ ]:
def _generate(messages, max_tokens=MAX_TOKENS, temperature=TEMPERATURE):
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = tokenizer.encode(text)
    reply  = generate_fn(
        model, tokenizer,
        prompt  = tokens,
        max_tokens = max_tokens,
        sampler = make_sampler_fn(temp=temperature, top_p=TOP_P),
        verbose = False,
    )
    return reply.strip()


def extract_aro_blocks(text):
    return [b.strip() for b in re.findall(r'```aro\n(.*?)```', text, re.DOTALL) if b.strip()]


# Heuristic: a reply is "checkable" only if at least one ARO block contains a
# feature-set header `(name: activity) {`. Q&A replies often include short
# fragments (e.g. two illustrative statements) that are valid ARO syntax in
# context but not a runnable program — running `aro check` on those produces
# a misleading FAIL. Skip them.
_FEATURESET_HEADER = re.compile(r'\(\s*[\w\- ]+\s*:\s*[^)]+\)\s*\{', re.DOTALL)


def is_complete_program(blocks):
    return any(_FEATURESET_HEADER.search(b) for b in blocks)


def aro_check(code):
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'check', tmp], capture_output=True, text=True, timeout=10)
            return r.returncode == 0, r.stdout + r.stderr
    except FileNotFoundError:
        return None, 'aro not found in PATH'
    except Exception as e:
        return None, str(e)


print('Helpers ready.')

## Interactive chat with teacher

- Type a question or a code request
- `quit`/`exit`/`q` to stop
- `reset` to clear history
- `check` to re-validate the last generated code

In [ ]:
history = [{'role': 'system', 'content': SYSTEM_PROMPT}]
last_reply = ''

print(f'teacher chat — {teacher_source}  (quit/exit/q to stop · reset to clear · check to validate)')
print('=' * 70)

while True:
    try:
        user_input = input('\nYou: ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\n[Interrupted]')
        break

    if not user_input:
        continue

    if user_input.lower() in ('quit', 'exit', 'q'):
        print('Goodbye.')
        break

    if user_input.lower() == 'reset':
        history = [{'role': 'system', 'content': SYSTEM_PROMPT}]
        last_reply = ''
        print('[History cleared]')
        continue

    if user_input.lower() == 'check':
        blocks = extract_aro_blocks(last_reply)
        if not blocks:
            print('[No ARO code blocks found in last reply]')
        elif not is_complete_program(blocks):
            print('[Last reply contains only ARO snippets, not a complete program — aro check skipped]')
        else:
            ok, output = aro_check('\n\n'.join(blocks))
            status = 'PASS' if ok is True else ('FAIL' if ok is False else '? (aro not in PATH)')
            print(f'[aro check: {status}]')
            if output.strip():
                print(output.strip())
        continue

    history.append({'role': 'user', 'content': user_input})

    print('\nTeacher: ', end='', flush=True)
    try:
        reply = _generate(history)
    except Exception as e:
        print(f'[Error: {e}]')
        history.pop()
        continue

    print(reply)
    last_reply = reply
    history.append({'role': 'assistant', 'content': reply})

    # Only run aro check when the reply contains a complete ARO program.
    # Q&A replies that include only ARO snippets (no feature-set header) are
    # left alone — checking a fragment would FAIL misleadingly.
    blocks = extract_aro_blocks(reply)
    if blocks and is_complete_program(blocks):
        ok, output = aro_check('\n\n'.join(blocks))
        if ok is True:
            print('\n[aro check: PASS]')
        elif ok is False:
            print(f'\n[aro check: FAIL]')
            if output.strip():
                print(output.strip())

## Probe set — quick teacher health check

Runs the same probe set as NB25's chat probe, so the teacher's pre-distillation
performance can be compared against the final packaged student. If the teacher
scores under ~50% on code probes here, distillation in NB21 will produce mostly
junk and is not worth running — fix the upstream SFT/DPO first.

In [ ]:
PROBES = [
    # Code generation
    ('code', 'Write a minimal ARO Application-Start that starts an HTTP server.'),
    ('code', 'Write an ARO feature set that retrieves a user by ID from a repository and returns it as an OK response.'),
    ('code', 'Write an ARO for-each loop that logs each item in a list.'),
    ('code', 'Write an ARO feature set that validates a request body and returns a 400 status on failure.'),
    ('code', 'Write an ARO Application-Start that reads a file and logs its contents.'),
    # Knowledge Q&A
    ('qa', 'How do I emit a custom event in ARO and handle it in another feature set?'),
    ('qa', 'Explain the difference between Extract and Compute in ARO.'),
    ('qa', 'How does the Publish action work in ARO? Give an example.'),
]

code_passed  = 0
code_failed  = 0
code_skipped = 0      # code probes whose reply had no complete ARO program
qa_answered  = 0
qa_empty     = 0

print(f'Running {len(PROBES)} probes against teacher ({teacher_source})...\n')

for i, (mode, prompt) in enumerate(PROBES):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]
    reply  = _generate(messages)
    blocks = extract_aro_blocks(reply)

    if mode == 'code':
        if not blocks:
            # Reply contains no aro code at all — don't run aro check, skip.
            status = 'no code (skip)'
            code_skipped += 1
        elif not is_complete_program(blocks):
            # Only fragments — running aro check would FAIL misleadingly.
            status = 'fragment (skip)'
            code_skipped += 1
        else:
            ok, _ = aro_check('\n\n'.join(blocks))
            if ok is True:
                status = 'pass'
                code_passed += 1
            elif ok is False:
                status = 'FAIL'
                code_failed += 1
            else:
                status = '? (aro not in PATH)'
    else:
        text_only = re.sub(r'```.*?```', '', reply, flags=re.DOTALL).strip()
        if len(text_only) > 30:
            status = 'answered'
            qa_answered += 1
        else:
            status = 'thin'
            qa_empty += 1

    print(f'  [{i+1:2d}] [{mode:4s}] {status:<18}  {prompt[:60]}')

total_code = code_passed + code_failed
total_qa   = qa_answered + qa_empty
print(f'\nCode probes:  {code_passed}/{total_code} syntax pass'
      + (f' ({100*code_passed/total_code:.0f}%)' if total_code else ''))
if code_skipped:
    print(f'              {code_skipped} skipped (reply had no complete program)')
print(f'Q&A probes:   {qa_answered}/{total_qa} answered'
      + (f' ({100*qa_answered/total_qa:.0f}%)' if total_qa else ''))

if total_code and code_passed / total_code < 0.5:
    print('\n  STRICT teacher health check FAILED.')
    print(f'  Teacher complete-program pass rate: {100*code_passed/total_code:.0f}%  (need >= 50%).')
    print('  Distillation in NB12 will inherit the teacher\'s defects and produce')
    print('  a broken student. Stop and re-run NB12/19/21 with stricter data filters')
    print('  (NB12 now drops fragment-only code samples) before running NB12.')
    raise SystemExit('NB24 strict probe failed — see above')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'text.color': '#222222', 'axes.labelcolor': '#222222',
    'xtick.color': '#333333', 'ytick.color': '#333333',
    'axes.titlecolor': '#111111', 'figure.facecolor': '#fafafa',
    'axes.facecolor': '#f9f9f9', 'savefig.facecolor': '#fafafa',
    'savefig.bbox': 'tight', 'savefig.dpi': 150,
})
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '24_chat_teacher.png'

total_code = code_passed + code_failed
total_qa   = qa_answered + qa_empty

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Code probes
_code_vals   = [code_passed, code_failed]
_code_labels = [f'Pass\n({code_passed})', f'Fail\n({code_failed})']
_code_colors = ['#2ecc71', '#e74c3c']
if total_code > 0:
    wedges, _, autotexts = ax1.pie(
        _code_vals, labels=_code_labels, colors=_code_colors,
        autopct='%1.0f%%', startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    )
    for t in autotexts: t.set_fontsize(11); t.set_fontweight('bold')
    ax1.add_artist(plt.Circle((0, 0), 0.5, fc='white'))
    ax1.text(0, 0, f'{100*code_passed/total_code:.0f}%', ha='center', va='center',
             fontsize=16, fontweight='bold')
else:
    ax1.text(0.5, 0.5, 'no code probes', ha='center', va='center', transform=ax1.transAxes)
ax1.set_title('Code Probes (aro check)', fontsize=12, fontweight='bold')

# Q&A probes
_qa_vals   = [qa_answered, qa_empty]
_qa_labels = [f'Answered\n({qa_answered})', f'Thin\n({qa_empty})']
_qa_colors = ['#3498db', '#f39c12']
if total_qa > 0:
    wedges, _, autotexts = ax2.pie(
        _qa_vals, labels=_qa_labels, colors=_qa_colors,
        autopct='%1.0f%%', startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    )
    for t in autotexts: t.set_fontsize(11); t.set_fontweight('bold')
    ax2.add_artist(plt.Circle((0, 0), 0.5, fc='white'))
    ax2.text(0, 0, f'{100*qa_answered/total_qa:.0f}%', ha='center', va='center',
             fontsize=16, fontweight='bold')
else:
    ax2.text(0.5, 0.5, 'no Q&A probes', ha='center', va='center', transform=ax2.transAxes)
ax2.set_title('Q&A Probes', fontsize=12, fontweight='bold')

fig.suptitle(
    f'NB24 Teacher Probes — {teacher_source}  ·  '
    f'{code_passed}/{total_code} code pass  ·  {qa_answered}/{total_qa} Q&A answered',
    fontsize=12, fontweight='bold', y=1.04,
)
fig.tight_layout()
fig.savefig(_out)
plt.close(fig)
print(f'Saved: {_out}')

## Next step

If probes look healthy, proceed to **NB21 (`21_distillation`)** to generate the student training data.
If they don't, re-run upstream training (NB17 SFT, NB18 preference-SFT, or NB20 iterative loop)
before spending an hour on distillation that will inherit the teacher's defects.